In [1]:
"""
Chest X-Ray Classification — EfficientNet-B4 & ViT-B/16 分離訓練 + Logit Ensemble
====================================================================================
環境：Kaggle GPU (T4 / P100)
問題：15-class single-label classification（~30,000 張標註圖）

改動紀錄（v2）：
  [A] 模型拆分：CNNModel / ViTModel 各自獨立訓練，最後 logit 加權平均
  [B] 取消 freeze warmup，改成全程差分 lr（資料充足，backbone 不易崩）
  [C] Augmentation 強化：
        ‣ GaussNoise / MultiplicativeNoise（感測器噪聲）
        ‣ CoarseDropout / GridDropout（Dropout 類遮蔽）
        ‣ ElasticTransform（醫學影像彈性形變）
        ‣ CutMix（training loop 層級，label 按面積混合）
  [D] val set 上 grid search 最佳 CNN/ViT alpha
  [E] Patient-based split、Checkpoint resume、ReduceLROnPlateau（沿用 v1）
"""

import os, re, random, math
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import GradScaler, autocast

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torchvision import datasets
from sklearn.metrics import roc_auc_score, classification_report, f1_score
from tqdm import tqdm

# ─────────────────────────────────────────────
# 0. 設定
# ─────────────────────────────────────────────
CFG = {
    "train_dir":  "/kaggle/input/datasets/e24126270/x-ray-dataset/xray_data/train",
    "test_dir":   "/kaggle/input/datasets/e24126270/x-ray-dataset/xray_data/test",
    "output_dir": "/kaggle/working",

    "num_classes": 15,
    "cnn_drop_rate": 0.3,
    "vit_drop_rate": 0.5,   # ViT 容易 overfit，dropout 拉高

    # 訓練尺寸
    "img_size":    224,
    "img_size_ft": 384,

    # batch size
    "batch_size":    20,
    "batch_size_ft": 8,

    # epoch（CNN / ViT 分開設定）
    "cnn_main_epochs": 20,   # CNN_P1 還在收斂，加長
    "cnn_ft_epochs":   3,    # CNN_P2 只需短暫 hr 適應
    "vit_main_epochs": 10,   # ViT epoch 5 就開始 overfit，設上限
    "vit_ft_epochs":   0,    # ViT_P2 直接跳過

    # early stopping patience（僅 ViT 使用）
    "early_stop_patience": 4,

    # optimizer（backbone / head 分開）
    "backbone_lr":    1e-5,
    "head_lr":        1e-4,
    "backbone_lr_ft": 5e-7,  # CNN_P2 lr 壓低，避免 val loss 發散
    "head_lr_ft":     5e-6,
    "cnn_weight_decay": 1e-2,
    "vit_weight_decay": 5e-2,  # ViT weight decay 拉大

    # patient split
    "patient_train_ratio": 0.9,

    "seed": 42,
    "num_workers": 4,
}

# ─────────────────────────────────────────────
# 1. 基礎設定
# ─────────────────────────────────────────────
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(CFG["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# ─────────────────────────────────────────────
# 2. Augmentation
#    [C] 加入噪聲、dropout、彈性形變
# ─────────────────────────────────────────────

def get_train_transforms(img_size: int) -> A.Compose:
    """
    訓練 augmentation pipeline。

    分層說明：
      Layer 1 - 幾何：Flip / ShiftScaleRotate（rotate ≤ 20°）/ Rotate（獨立小角度）
      Layer 2 - 像素噪聲：GaussNoise（感測器噪聲）
      Layer 3 - 模糊：GaussianBlur
      Layer 4 - 亮度對比：RandomBrightnessContrast
      Layer 5 - 彈性形變：ElasticTransform（醫學影像專用）
    """
    return A.Compose([
        # ── resize + 幾何 ──────────────────────────────────
        A.Resize(img_size + 32, img_size + 32,
                 interpolation=2),           # INTER_AREA，大比例縮放較不失真
        A.RandomCrop(img_size, img_size),
        A.HorizontalFlip(p=0.5),
        A.ShiftScaleRotate(
            shift_limit=0.06, scale_limit=0.12,
            rotate_limit=20, p=0.6,          # ← 最大旋轉角改為 20°
        ),
        # 額外的純旋轉，讓小角度擾動更充分
        A.Rotate(limit=20, p=0.3),

        # ── Layer 2：像素噪聲 ──────────────────────────────
        A.GaussNoise(var_limit=(10.0, 50.0), p=0.4),

        # ── Layer 3：模糊 ───────────────────────────────────
        A.GaussianBlur(blur_limit=(3, 5), p=0.15),

        # ── Layer 4：亮度 / 對比 ────────────────────────────
        A.RandomBrightnessContrast(
            brightness_limit=0.25, contrast_limit=0.25, p=0.4,
        ),

        # ── Layer 5：ElasticTransform（醫學影像彈性形變）─────
        # 模擬不同呼吸深度 / 體位造成的器官形變
        A.ElasticTransform(
            alpha=120, sigma=6, alpha_affine=3.6,
            p=0.25,
        ),

        # ── 標準化 + ToTensor ───────────────────────────────
        A.Normalize(mean=(0.485, 0.456, 0.406),
                    std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])


def get_val_transforms(img_size: int) -> A.Compose:
    return A.Compose([
        A.Resize(img_size, img_size, interpolation=2),
        A.Normalize(mean=(0.485, 0.456, 0.406),
                    std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])


def get_tta_transforms(img_size: int) -> list:
    """4× TTA：原圖 / HFlip / CenterCrop / CenterCrop+HFlip"""
    mean, std = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
    sz = img_size
    return [
        A.Compose([A.Resize(sz, sz, 2),
                   A.Normalize(mean, std), ToTensorV2()]),
        A.Compose([A.Resize(sz, sz, 2), A.HorizontalFlip(p=1.0),
                   A.Normalize(mean, std), ToTensorV2()]),
        A.Compose([A.Resize(sz+32, sz+32, 2), A.CenterCrop(sz, sz),
                   A.Normalize(mean, std), ToTensorV2()]),
        A.Compose([A.Resize(sz+32, sz+32, 2), A.CenterCrop(sz, sz),
                   A.HorizontalFlip(p=1.0),
                   A.Normalize(mean, std), ToTensorV2()]),
    ]

# ─────────────────────────────────────────────
# 4. Dataset
# ─────────────────────────────────────────────

_PATIENT_RE = re.compile(r"^(\d+)_\d+\.")

def _get_patient_id(filename: str) -> str:
    m = _PATIENT_RE.match(Path(filename).name)
    return m.group(1) if m else Path(filename).stem


class XRayDataset(Dataset):
    def __init__(self, samples: list, num_classes: int, transform=None):
        self.samples     = samples
        self.num_classes = num_classes
        self.transform   = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = np.array(Image.open(path).convert("RGB"))
        if self.transform:
            image = self.transform(image=image)["image"]
        target = torch.zeros(self.num_classes, dtype=torch.float32)
        target[label] = 1.0
        return image, target


class TestDataset(Dataset):
    def __init__(self, test_dir: str, transform=None):
        self.paths     = sorted(Path(test_dir).glob("*.*g"))
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        path  = self.paths[idx]
        image = np.array(Image.open(path).convert("RGB"))
        if self.transform:
            image = self.transform(image=image)["image"]
        return image, path.name

# ─────────────────────────────────────────────
# 5. Patient-based split
# ─────────────────────────────────────────────

def patient_split(train_dir: str, train_ratio: float = 0.9, seed: int = 42):
    full_ds     = datasets.ImageFolder(train_dir)
    class_names = full_ds.classes

    patient_dict = defaultdict(list)
    for path, label in full_ds.samples:
        pid = _get_patient_id(Path(path).name)
        patient_dict[pid].append((path, label))

    patients = list(patient_dict.keys())
    random.Random(seed).shuffle(patients)
    cut = int(len(patients) * train_ratio)

    train_samples = [s for pid in patients[:cut] for s in patient_dict[pid]]
    val_samples   = [s for pid in patients[cut:] for s in patient_dict[pid]]

    print(f"\n[Patient Split]  {len(patients)} 病人  |  "
          f"train {len(patients[:cut])} ({len(train_samples)} 張)  "
          f"val {len(patients[cut:])} ({len(val_samples)} 張)")
    print(f"Classes: {class_names}")
    return train_samples, val_samples, class_names


def build_dataloaders(img_size: int, batch_size: int):
    train_samples, val_samples, class_names = patient_split(
        CFG["train_dir"], CFG["patient_train_ratio"], CFG["seed"])

    train_ds = XRayDataset(train_samples, CFG["num_classes"],
                           transform=get_train_transforms(img_size))
    val_ds   = XRayDataset(val_samples,   CFG["num_classes"],
                           transform=get_val_transforms(img_size))

    train_loader = DataLoader(train_ds, batch_size=batch_size,
                              shuffle=True,  num_workers=CFG["num_workers"],
                              pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size * 2,
                              shuffle=False, num_workers=CFG["num_workers"],
                              pin_memory=True)

    # pos_weight
    label_counts = np.zeros(CFG["num_classes"], dtype=float)
    for _, lbl in train_samples:
        label_counts[lbl] += 1
    total = label_counts.sum()
    pos_weight = torch.tensor(
        (total - label_counts) / np.where(label_counts == 0, 1.0, label_counts),
        dtype=torch.float32,
    ).to(device)

    return train_loader, val_loader, class_names, pos_weight

# ─────────────────────────────────────────────
# 6. 模型定義（[A] 拆成兩個獨立模型）
# ─────────────────────────────────────────────

def _make_head(in_dim: int, num_classes: int, drop_rate: float) -> nn.Sequential:
    return nn.Sequential(
        nn.LayerNorm(in_dim),
        nn.Dropout(drop_rate),
        nn.Linear(in_dim, 512),
        nn.GELU(),
        nn.Dropout(drop_rate / 2),
        nn.Linear(512, num_classes),
    )


class CNNModel(nn.Module):
    """EfficientNet-B4 + custom head"""
    def __init__(self, num_classes: int = 15, drop_rate: float = 0.3):
        super().__init__()
        self.backbone = timm.create_model(
            "efficientnet_b4", pretrained=True,
            num_classes=0, global_pool="avg",
        )
        self.head = _make_head(self.backbone.num_features, num_classes, drop_rate)

    def forward(self, x):
        return self.head(self.backbone(x))

    def get_optimizer_groups(self, backbone_lr, head_lr):
        return [
            {"params": self.backbone.parameters(), "lr": backbone_lr},
            {"params": self.head.parameters(),     "lr": head_lr},
        ]


class ViTModel(nn.Module):
    """ViT-B/16 + custom head"""
    def __init__(self, num_classes: int = 15, drop_rate: float = 0.3,
                 img_size: int = 224):
        super().__init__()
        self.backbone = timm.create_model(
            "vit_base_patch16_224", pretrained=True,
            num_classes=0, img_size=img_size,
            dynamic_img_size=True,
        )
        self.head = _make_head(self.backbone.num_features, num_classes, drop_rate)

    def forward(self, x):
        return self.head(self.backbone(x))

    def get_optimizer_groups(self, backbone_lr, head_lr):
        return [
            {"params": self.backbone.parameters(), "lr": backbone_lr},
            {"params": self.head.parameters(),     "lr": head_lr},
        ]

# ─────────────────────────────────────────────
# 7. Train / Validate
# ─────────────────────────────────────────────

def train_one_epoch(model, loader, criterion, optimizer, scaler,
                    scheduler=None):
    model.train()
    total_loss, total = 0.0, 0

    for images, labels in tqdm(loader, leave=False, desc="  train"):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        with autocast("cuda"):
            loss = criterion(model(images), labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        if scheduler:
            scheduler.step()

        total_loss += loss.item() * images.size(0)
        total      += images.size(0)

    return total_loss / total


@torch.no_grad()
def validate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    all_labels, all_probs = [], []

    for images, labels in tqdm(loader, leave=False, desc="  val  "):
        images, labels = images.to(device), labels.to(device)
        with autocast("cuda"):
            logits = model(images)
            loss   = criterion(logits, labels)
        total_loss += loss.item() * images.size(0)
        all_probs.append(torch.sigmoid(logits).cpu().numpy())
        all_labels.append(labels.cpu().numpy())

    all_labels = np.concatenate(all_labels)
    all_probs  = np.concatenate(all_probs)

    try:
        auc = roc_auc_score(all_labels, all_probs, average="macro")
    except ValueError:
        auc = float("nan")

    return total_loss / len(all_labels), auc, all_labels, all_probs


def tune_thresholds(all_labels, all_probs, num_classes):
    thresholds = np.full(num_classes, 0.5)
    for c in range(num_classes):
        best_f1, best_t = 0.0, 0.5
        for t in np.arange(0.05, 0.96, 0.05):
            preds = (all_probs[:, c] >= t).astype(int)
            f1 = f1_score(all_labels[:, c], preds, zero_division=0)
            if f1 > best_f1:
                best_f1, best_t = f1, t
        thresholds[c] = best_t
    return thresholds

# ─────────────────────────────────────────────
# 8. Checkpoint resume helper
# ─────────────────────────────────────────────

def find_resume_epoch(out: Path, tag: str) -> int:
    pattern = re.compile(rf"ckpt_{tag}_ep(\d+)\.pth")
    epochs_done = [int(m.group(1))
                   for f in out.iterdir() if (m := pattern.match(f.name))]
    return max(epochs_done) if epochs_done else 0

# ─────────────────────────────────────────────
# 9. 通用訓練 phase
# ─────────────────────────────────────────────

def run_phase(tag, model, train_loader, val_loader,
              criterion, optimizer, cosine_scheduler, plateau_scheduler,
              scaler, epochs, best_auc, best_path, out: "Path",
              start_epoch: int = 0, early_stop_patience: int = 0):
    """
    early_stop_patience > 0 時啟用 early stopping（監控 val AUC）。
    設為 0（預設）則跑滿 epochs。
    """
    no_improve = 0

    for epoch in range(start_epoch + 1, epochs + 1):
        tr_loss = train_one_epoch(
            model, train_loader, criterion, optimizer, scaler,
        )
        vl_loss, vl_auc, _, _ = validate(model, val_loader, criterion)

        cosine_scheduler.step()
        plateau_scheduler.step(vl_loss)
        cur_lr = optimizer.param_groups[0]["lr"]

        print(f"[{tag} {epoch:02d}/{epochs}]  "
              f"tr={tr_loss:.4f}  vl={vl_loss:.4f}  auc={vl_auc:.4f}  "
              f"lr={cur_lr:.2e}")

        if vl_auc > best_auc:
            best_auc  = vl_auc
            no_improve = 0
            torch.save(model.state_dict(), best_path)
            print(f"  ✓ Best saved  AUC={best_auc:.4f}")
        else:
            no_improve += 1
            if early_stop_patience > 0 and no_improve >= early_stop_patience:
                print(f"  ✗ Early stop at epoch {epoch} "
                      f"(no improvement for {early_stop_patience} epochs)")
                break

        # epoch checkpoint（用於 resume）
        torch.save(model.state_dict(), out / f"ckpt_{tag}_ep{epoch}.pth")

    return best_auc


# ─────────────────────────────────────────────
# 10. [D] Logit alpha grid search
# ─────────────────────────────────────────────

def search_alpha(cnn_logits: np.ndarray, vit_logits: np.ndarray,
                 val_labels: np.ndarray) -> float:
    """
    在 val set 上搜尋最佳 CNN/ViT 混合比例。
    cnn_logits, vit_logits : (N, C)  raw logits（未過 sigmoid）
    val_labels             : (N, C)  one-hot float
    """
    best_alpha, best_auc = 0.5, 0.0
    print("\n[Alpha Search]")
    for alpha in np.arange(0.20, 0.85, 0.05):
        mixed  = alpha * cnn_logits + (1 - alpha) * vit_logits
        probs  = 1 / (1 + np.exp(-mixed))          # sigmoid
        try:
            auc = roc_auc_score(val_labels, probs, average="macro")
        except ValueError:
            auc = float("nan")
        marker = " ← best" if auc > best_auc else ""
        print(f"  alpha(cnn)={alpha:.2f}  AUC={auc:.4f}{marker}")
        if auc > best_auc:
            best_auc, best_alpha = auc, alpha

    print(f"\n★ Best alpha(cnn)={best_alpha:.2f}  AUC={best_auc:.4f}")
    return best_alpha


# ─────────────────────────────────────────────
# 11. TTA 推理
# ─────────────────────────────────────────────

@torch.no_grad()
def get_test_logits_tta(model, test_dir: str, img_size: int) -> tuple:
    """回傳 (fnames, logits_array(N, C))"""
    model.eval()
    tfs   = get_tta_transforms(img_size)
    paths = sorted(Path(test_dir).glob("*.*g"))
    all_logits, fnames = [], []

    for path in tqdm(paths, desc="  TTA"):
        image     = np.array(Image.open(path).convert("RGB"))
        tta_logits = []
        for tf in tfs:
            t = tf(image=image)["image"].unsqueeze(0).to(device)
            with autocast("cuda"):
                tta_logits.append(model(t).squeeze(0).cpu().numpy())
        all_logits.append(np.mean(tta_logits, axis=0))
        fnames.append(path.name)

    return fnames, np.array(all_logits)   # (N, C)


def make_submission(fnames, probs, class_names, thresholds) -> pd.DataFrame:
    binary = (probs >= thresholds[np.newaxis, :]).astype(int)
    no_pred = binary.sum(axis=1) == 0
    binary[no_pred, probs[no_pred].argmax(axis=1)] = 1

    pred_name = [sorted([class_names[i] for i, v in enumerate(row) if v == 1])[-1]
                 for row in binary]

    df = pd.DataFrame({"filename": fnames, "label": pred_name})
    df["_num"] = df["filename"].str.extract(r"(\d+)").astype(int)
    df = df.sort_values("_num").reset_index(drop=True)
    df.insert(0, "id", df.index)
    return df.drop(columns=["_num"])


# ─────────────────────────────────────────────
# 12. 單一模型完整訓練流程
# ─────────────────────────────────────────────

def train_model(model_name: str, model, train_loader, val_loader,
                criterion, scaler, out: Path):
    """
    Phase 1：224px  main training
    Phase 2：384px  high-res fine-tune（CNN only；ViT_ft_epochs=0 時跳過）

    CNN / ViT 使用各自的 epoch、weight_decay、early_stop 設定。
    """
    is_vit      = isinstance(model, ViTModel)
    main_epochs = CFG["vit_main_epochs"] if is_vit else CFG["cnn_main_epochs"]
    ft_epochs   = CFG["vit_ft_epochs"]   if is_vit else CFG["cnn_ft_epochs"]
    wd          = CFG["vit_weight_decay"] if is_vit else CFG["cnn_weight_decay"]
    es_patience = CFG["early_stop_patience"] if is_vit else 0

    best_path = out / f"best_{model_name}.pth"
    best_auc  = 0.0

    # ── Phase 1：224px ──────────────────────────────────────
    print(f"\n{'='*55}")
    print(f"[{model_name}] Phase 1 — 224px  ({main_epochs} epochs"
          + (f", early_stop patience={es_patience}" if es_patience else "") + ")")
    print('='*55)

    done = find_resume_epoch(out, f"{model_name}_P1")
    opt1 = torch.optim.AdamW(
        model.get_optimizer_groups(CFG["backbone_lr"], CFG["head_lr"]),
        weight_decay=wd,
    )
    cos1 = torch.optim.lr_scheduler.CosineAnnealingLR(
        opt1, T_max=main_epochs, eta_min=1e-6,
    )
    plat1 = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt1, mode="min", factor=0.1, patience=4, min_lr=1e-7,
    )

    for _ in range(done):
        cos1.step()

    if best_path.exists():
        model.load_state_dict(torch.load(best_path, weights_only=True))

    best_auc = run_phase(
        f"{model_name}_P1", model, train_loader, val_loader,
        criterion, opt1, cos1, plat1, scaler,
        main_epochs, best_auc, best_path, out,
        start_epoch=done, early_stop_patience=es_patience,
    )

    # ── Phase 2：384px（ft_epochs=0 時直接跳過）──────────────
    if ft_epochs == 0:
        print(f"\n[{model_name}] Phase 2 skipped (ft_epochs=0)")
        return model, best_auc

    print(f"\n{'='*55}")
    print(f"[{model_name}] Phase 2 — 384px  ({ft_epochs} epochs)")
    print('='*55)

    train_loader_hr, val_loader_hr, _, _ = \
        build_dataloaders(CFG["img_size_ft"], CFG["batch_size_ft"])

    if is_vit:
        model_hr = ViTModel(CFG["num_classes"], CFG["vit_drop_rate"],
                            img_size=CFG["img_size_ft"]).to(device)
        ckpt = torch.load(best_path, weights_only=True)
        ckpt.pop("backbone.pos_embed", None)
        model_hr.load_state_dict(ckpt, strict=False)
    else:
        model_hr = model
        model_hr.load_state_dict(
            torch.load(best_path, weights_only=True))

    done_hr = find_resume_epoch(out, f"{model_name}_P2")
    opt2    = torch.optim.AdamW(
        model_hr.get_optimizer_groups(CFG["backbone_lr_ft"], CFG["head_lr_ft"]),
        weight_decay=wd,
    )
    cos2  = torch.optim.lr_scheduler.CosineAnnealingLR(
        opt2, T_max=ft_epochs, eta_min=1e-7,
    )
    plat2 = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt2, mode="min", factor=0.1, patience=2, min_lr=1e-8,
    )
    for _ in range(done_hr):
        cos2.step()

    best_auc = run_phase(
        f"{model_name}_P2", model_hr, train_loader_hr, val_loader_hr,
        criterion, opt2, cos2, plat2, scaler,
        ft_epochs, best_auc, best_path, out,
        start_epoch=done_hr,
    )

    return model_hr, best_auc


# ─────────────────────────────────────────────
# 13. 主流程
# ─────────────────────────────────────────────

def main():
    out = Path(CFG["output_dir"])
    out.mkdir(parents=True, exist_ok=True)

    train_loader, val_loader, class_names, pos_weight = \
        build_dataloaders(CFG["img_size"], CFG["batch_size"])

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    scaler    = GradScaler("cuda")

    # ── 訓練 CNN ──────────────────────────────────────────
    cnn = CNNModel(CFG["num_classes"], CFG["cnn_drop_rate"]).to(device)
    cnn_final, _ = train_model(
        "CNN", cnn, train_loader, val_loader, criterion, scaler, out)

    # ── 訓練 ViT ──────────────────────────────────────────
    vit = ViTModel(CFG["num_classes"], CFG["vit_drop_rate"]).to(device)
    vit_final, _ = train_model(
        "ViT", vit, train_loader, val_loader, criterion, scaler, out)

    # ── [D] 在 val set 上收集 logits，搜尋最佳 alpha ─────
    print("\n[Collecting val logits for alpha search ...]")
    # 重新載入最佳權重
    cnn_best_path = out / "best_CNN.pth"
    vit_best_path = out / "best_ViT.pth"

    # 用 384px val loader（最後階段的 val）
    _, val_loader_hr, _, _ = build_dataloaders(
        CFG["img_size_ft"], CFG["batch_size_ft"])

    @torch.no_grad()
    def collect_logits(model, loader):
        model.eval()
        all_logits, all_labels = [], []
        for images, labels in tqdm(loader, desc="  collect", leave=False):
            images = images.to(device)
            with autocast("cuda"):
                all_logits.append(model(images).cpu().numpy())
            all_labels.append(labels.numpy())
        return np.concatenate(all_logits), np.concatenate(all_labels)

    # 重建 ViT (384px pos_embed) 並載入
    vit_hr = ViTModel(CFG["num_classes"], CFG["vit_drop_rate"],
                      img_size=CFG["img_size_ft"]).to(device)
    ckpt_vit = torch.load(vit_best_path, weights_only=True)
    ckpt_vit.pop("backbone.pos_embed", None)
    vit_hr.load_state_dict(ckpt_vit, strict=False)

    cnn_final.load_state_dict(torch.load(cnn_best_path, weights_only=True))

    cnn_logits, val_labels = collect_logits(cnn_final, val_loader_hr)
    vit_logits, _          = collect_logits(vit_hr,    val_loader_hr)

    alpha = search_alpha(cnn_logits, vit_logits, val_labels)

    # ── Threshold tuning（用最佳 alpha 混合的 prob）──────
    mixed_probs = 1 / (1 + np.exp(
        -(alpha * cnn_logits + (1 - alpha) * vit_logits)))
    thresholds = tune_thresholds(val_labels, mixed_probs, CFG["num_classes"])
    print("\nFinal thresholds:", dict(zip(class_names, thresholds.round(2))))

    # Val report
    binary_preds = (mixed_probs >= thresholds[np.newaxis, :]).astype(int)
    no_pred = binary_preds.sum(axis=1) == 0
    binary_preds[no_pred, mixed_probs[no_pred].argmax(axis=1)] = 1
    print("\nPer-class Classification Report:")
    print(classification_report(val_labels.astype(int), binary_preds,
                                 target_names=class_names, zero_division=0))

    # ── TTA 推理 → submission ─────────────────────────────
    print("\n" + "="*55)
    print("Generating submission (TTA × 4, ensemble)")
    print("="*55)
    fnames_c, cnn_test_logits = get_test_logits_tta(
        cnn_final, CFG["test_dir"], CFG["img_size_ft"])
    fnames_v, vit_test_logits = get_test_logits_tta(
        vit_hr,    CFG["test_dir"], CFG["img_size_ft"])

    assert fnames_c == fnames_v, "CNN/ViT test file order mismatch"
    test_probs = 1 / (1 + np.exp(
        -(alpha * cnn_test_logits + (1 - alpha) * vit_test_logits)))

    sub = make_submission(fnames_c, test_probs, class_names, thresholds)
    sub_path = out / "submission.csv"
    sub.to_csv(sub_path, index=False)
    print(f"Saved → {sub_path}")
    print(sub.head(10))

    # 記錄 alpha
    pd.DataFrame([{"alpha_cnn": alpha, "alpha_vit": 1 - alpha}]).to_csv(
        out / "ensemble_config.csv", index=False)


if __name__ == "__main__":
    main()


Device: cuda

[Patient Split]  7824 病人  |  train 7041 (20683 張)  val 783 (2238 張)
Classes: ['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Effusion', 'Emphysema', 'Fibrosis', 'Hernia', 'Infiltration', 'Mass', 'No Finding', 'Nodule', 'Pleural_Thickening', 'Pneumonia', 'Pneumothorax']


/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
/tmp/ipykernel_22/3005172383.py:128: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(10.0, 50.0), p=0.4),
/tmp/ipykernel_22/3005172383.py:140: UserWarning: Argument(s) 'alpha_affine' are not valid for transform ElasticTransform
  A.ElasticTransform(


model.safetensors:   0%|          | 0.00/77.9M [00:00<?, ?B/s]


[CNN] Phase 1 — 224px  (20 epochs)


[CNN_P1 01/20]  tr=1.3340  vl=1.3613  auc=0.6694  lr=9.94e-06
  ✓ Best saved  AUC=0.6694


...

[ViT_P1 10/10]  tr=1.0301  vl=1.2323  auc=0.7791  lr=1.00e-06

[ViT] Phase 2 skipped (ft_epochs=0)

[Collecting val logits for alpha search ...]

[Patient Split]  7824 病人  |  train 7041 (20683 張)  val 783 (2238 張)
Classes: ['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Effusion', 'Emphysema', 'Fibrosis', 'Hernia', 'Infiltration', 'Mass', 'No Finding', 'Nodule', 'Pleural_Thickening', 'Pneumonia', 'Pneumothorax']


/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
/tmp/ipykernel_22/3005172383.py:128: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(10.0, 50.0), p=0.4),
/tmp/ipykernel_22/3005172383.py:140: UserWarning: Argument(s) 'alpha_affine' are not valid for transform ElasticTransform
  A.ElasticTransform(



[Alpha Search]
  alpha(cnn)=0.20  AUC=0.7805 ← best
  alpha(cnn)=0.25  AUC=0.7812 ← best
  alpha(cnn)=0.30  AUC=0.7815 ← best
  alpha(cnn)=0.35  AUC=0.7816 ← best
  alpha(cnn)=0.40  AUC=0.7812
  alpha(cnn)=0.45  AUC=0.7806
  alpha(cnn)=0.50  AUC=0.7794
  alpha(cnn)=0.55  AUC=0.7776
  alpha(cnn)=0.60  AUC=0.7755
  alpha(cnn)=0.65  AUC=0.7729
  alpha(cnn)=0.70  AUC=0.7697
  alpha(cnn)=0.75  AUC=0.7656
  alpha(cnn)=0.80  AUC=0.7608

★ Best alpha(cnn)=0.35  AUC=0.7816

Final thresholds: {'Atelectasis': np.float64(0.55), 'Cardiomegaly': np.float64(0.6), 'Consolidation': np.float64(0.65), 'Edema': np.float64(0.8), 'Effusion': np.float64(0.7), 'Emphysema': np.float64(0.75), 'Fibrosis': np.float64(0.7), 'Hernia': np.float64(0.2), 'Infiltration': np.float64(0.5), 'Mass': np.float64(0.7), 'No Finding': np.float64(0.65), 'Nodule': np.float64(0.7), 'Pleural_Thickening': np.float64(0.6), 'Pneumonia': np.float64(0.5), 'Pneumothorax': np.float64(0.75)}

Per-class Classification Report:
             

  TTA: 100%|██████████| 5000/5000 [05:25<00:00, 15.36it/s]


Saved → /kaggle/working/submission.csv
   id filename               label
0   0    0.jpg  Pleural_Thickening
1   1    1.jpg                Mass
2   2    2.jpg            Effusion
3   3    3.jpg              Nodule
4   4    4.jpg            Effusion
5   5    5.jpg          No Finding
6   6    6.jpg  Pleural_Thickening
7   7    7.jpg          No Finding
8   8    8.jpg              Nodule
9   9    9.jpg            Effusion
